# CodaMOSA vs CoverUp — Experiment Control

This notebook is the single observable control panel. It calls versioned scripts; it does not duplicate the experiment logic.

**Design interpretation:** File B wins. This repository runs CodaMOSA and CoverUp on MT and CM with B1/B2. Pynguin and MuTAP rows come from the collaborator. Full R=10 execution stays locked until the R=2 pilot is complete and explicitly approved.

**Safety:** edit `config/local.env`, which is gitignored. Never paste keys into notebook cells or `experiment.toml`.

**Kernel:** use `.venv (Python 3.14.4)`. This notebook must be opened from inside the repository directory.`

## 0. Kernel and local credentials

Use `.venv (Python 3.14.4)`. Fill `config/local.env` outside this notebook. This cell prints presence only, never values.`

In [10]:
from pathlib import Path
import sys

# Find the repo root by walking up from the current directory.
_root = Path.cwd()
while _root != _root.parent:
    if (_root / "config" / "experiment.toml").exists():
        break
    _root = _root.parent
ROOT = _root

if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from notebook_control import load_local_env, presence, run, print_status

loaded = load_local_env()
print("Loaded variable names:", loaded)
print("Presence:", presence())
assert all(presence().values()), "Fill config/local.env before API probes."

Loaded variable names: ['LLM_BASE_URL', 'LLM_API_KEY', 'LLM_BASE_URL_B2', 'LLM_API_KEY_B2']
Presence: {'LLM_API_KEY': True, 'LLM_BASE_URL': True, 'LLM_API_KEY_B2': True, 'LLM_BASE_URL_B2': True}


## 1. No-cost setup and validation

Creates/updates the environment, inventories 588 tasks, builds the 64-run pilot matrix, and runs tests. Safe to rerun.`

In [11]:
run(["bash", "scripts/setup.sh"], timeout=1200)

$ bash scripts/setup.sh
Processing ./coverup
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for CoverUp: filename=coverup-0.6.3-py3-none-any.whl size=46068 sha256=7a8d86dbb396002c3c27ccfda8f2fe5c447bf508d7fb9864f8edf39fa57ec12a
  Stored in directory: /tmp/pip-ephem-wheel-cache-tllzoyhf/wheels/7a/13/08/9d65700f309d2c618b7b1b0d524d20c56a305cbc2f22ba4ba0
Successfully built CoverUp
  Attempting uninstall: CoverUp
    Found existing installation: CoverUp 0.6.3
    Uninstalling CoverUp-0.6.3:
      Successfully uninstalled CoverUp-0.6.3
Installed kernelspec teori-experiment in /home/masoyi/.local/share/jupyter/kernels/teori-experiment
MT tasks: 163
CM tasks: 425
Total available tasks: 588
Wrote 64 

CompletedProcess(args=['bash', 'scripts/setup.sh'], returncode=0, stdout="Requirement already satisfied: pip in ./.venv/lib/python3.14/site-packages (26.2.2)\nRequirement already satisfied: coverage==7.16.2 in ./.venv/lib/python3.14/site-packages (from -r requirements.lock (line 1)) (7.16.2)\nRequirement already satisfied: cosmic-ray==8.7.0 in ./.venv/lib/python3.14/site-packages (from -r requirements.lock (line 2)) (8.7.0)\nRequirement already satisfied: httpx==0.28.1 in ./.venv/lib/python3.14/site-packages (from -r requirements.lock (line 3)) (0.28.1)\nRequirement already satisfied: matplotlib==3.11.2 in ./.venv/lib/python3.14/site-packages (from -r requirements.lock (line 4)) (3.11.2)\nRequirement already satisfied: numpy==2.5.3 in ./.venv/lib/python3.14/site-packages (from -r requirements.lock (line 5)) (2.5.3)\nRequirement already satisfied: pandas==3.0.6 in ./.venv/lib/python3.14/site-packages (from -r requirements.lock (line 6)) (3.0.6)\nRequirement already satisfied: pytest==9.

## 2. Inspect dataset inventory and pilot plan

Expected totals: 163 MT functions + 425 CM modules = 588. Pilot: 5 MT + 3 disjoint CM × 2 tools × 2 backbones × R=2 = 64 runs.`

In [12]:
run([sys.executable, "scripts/inventory.py"])
run([sys.executable, "scripts/build_matrix.py", "--pilot"])
run([sys.executable, "scripts/run_experiment.py", "--pilot", "--dry-run"])
print_status()

$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/inventory.py
MT tasks: 163
CM tasks: 425
Total available tasks: 588

exit=0 elapsed=0.9s
$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/build_matrix.py --pilot
Wrote 64 runs to results/pilot_matrix.csv

exit=0 elapsed=0.0s
$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/run_experiment.py --pilot --dry-run
{
  "matrix": "/home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/results/pilot_matrix.csv",
  "results": "/home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/results/pilot_results.csv",
  "total": 64,
  "completed_ok": 0,
  "recorded": 64,
  "pending": 0,
  "cost_usd": 3.370008
}

exit=0 elapsed=0.4s
{
  "credentials": {
    "LLM_API_KEY": true,
    "LLM_BASE_URL": true,
    "LLM_API_KEY_B2": true,
    "LLM_BASE_URL_B2": true
  },
  "files": {
    "pilot_matrix.csv": {
      "rows": 64,
      "ok": 0,
      "failed": 0,
   

In [13]:
run([sys.executable, "scripts/report.py"])
from IPython.display import Markdown
Markdown((ROOT / "experiment_report.md").read_text(encoding="utf-8"))

$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/report.py
Wrote Markdown report to /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/experiment_report.md

exit=0 elapsed=0.1s


# Laporan Eksperimen CodaMOSA vs CoverUp

## 1. Status

Dokumen ini dibuat otomatis dari artefak pipeline. Angka hasil hanya diisi dari data yang benar-benar diukur; angka dummy File A tidak digunakan sebagai target atau hasil.

- Probe endpoint: **selesai**
- Pilot R=2: **selesai/sebagian**
- Run utama R=10: **belum dijalankan**

## 2. Interpretasi Rancangan

File B bersifat otoritatif. Pipeline ini mengukur CodaMOSA dan CoverUp pada MT dan CM dengan dua backbone. Hasil Pynguin dan MuTAP akan digabung dari kolaborator menggunakan skema hasil yang sama.

Perbandingan tool pada suite yang sama bersifat berpasangan per unit. Perbandingan MT dengan CM untuk tool yang sama bersifat deskriptif karena unit MT dan CM tidak berpasangan.

## 3. Inventaris Dataset

| Suite | Definisi satu task | Tersedia | Dipakai pilot | Dipakai run utama |
|---|---|---|---|---|
| MT | Satu fungsi benchmark dalam satu modul f001–f163 | 163 | 5 | 163 |
| CM | Satu modul Python pada satu baris cm_modules.csv | 425 | 3 | 40 |
| Total | — | 588 | 8 | 203 |

Inventaris mentah berisi **163 MT + 425 CM = 588 task**. Sesuai File B, run utama tidak memakai seluruh CM; ia memakai seluruh 163 MT dan sampel CM bertingkat sebanyak 40, sehingga terdapat **203 unit analisis utama**.

### Sampel CM per proyek

| Proyek | Jumlah |
|---|---|
| PySnooper | 1 |
| ansible | 10 |
| black | 1 |
| dataclasses-json | 1 |
| flutils | 1 |
| httpie | 3 |
| isort | 1 |
| py-backwards | 2 |
| pyMonet | 1 |
| pypara | 1 |
| python-semantic-release | 1 |
| pytutils | 1 |
| sty | 1 |
| tornado | 1 |
| typesystem | 4 |
| youtube-dl | 10 |

### Sampel CM per stratum

| Stratum | Jumlah |
|---|---|
| L0B0 | 6 |
| L0B1 | 6 |
| L1B0 | 5 |
| L1B1 | 5 |
| L1B2 | 4 |
| L2B0 | 4 |
| L2B1 | 5 |
| L2B2 | 5 |

Deviasi yang disetujui: stratum L0B2 kosong pada populasi aktual, sehingga sampel mencakup seluruh delapan stratum non-kosong, bukan memaksakan stratum kesembilan.

## 4. Matriks Run

| Tahap | Jumlah run |
|---|---|
| Pilot: 5 MT + 3 CM × 2 tool × 2 backbone × R=2 | 64 |
| Utama: 163 MT + 40 CM × 2 tool × 2 backbone × R=10 | 8120 |

## 5. Model dan Probe

| Backbone | Requested | Returned | Function calling | Status/Error |
|---|---|---|---|---|
| B1 | entah/auto | auto | True | PASS |
| B2 | entah/auto | auto | True | PASS |

B1 dilaporkan sebagai `gpt-5.6-sol (seller-claimed; endpoint reports "auto")`. Identitas model adalah klaim penjual dan tidak dapat diverifikasi. Field model mentah dari setiap respons harus tetap disimpan untuk audit routing.

## 6. Hasil Pilot

| Suite | Tool | Backbone | Run tercatat | OK | Gagal | Biaya USD |
|---|---|---|---|---|---|---|
| CM | CodaMOSA | B1 | 6 | 0 | 6 | 0.000000 |
| CM | CodaMOSA | B2 | 6 | 0 | 6 | 0.000000 |
| CM | CoverUp | B1 | 6 | 0 | 6 | 1.130484 |
| CM | CoverUp | B2 | 6 | 0 | 6 | 0.848164 |
| MT | CodaMOSA | B1 | 10 | 0 | 10 | 0.000000 |
| MT | CodaMOSA | B2 | 10 | 0 | 10 | 0.000000 |
| MT | CoverUp | B1 | 10 | 0 | 10 | 0.783260 |
| MT | CoverUp | B2 | 10 | 0 | 10 | 0.608100 |

## 7. Hasil Utama

_Belum tersedia._

Analisis akhir menggunakan mean, median, dan standard deviation; Wilcoxon signed-rank berpasangan; koreksi Holm; Vargha–Delaney A12; dan bootstrap CI 95% untuk selisih mean.

## 8. Task Gagal atau Dikeluarkan

Daftar rinci tersedia di `results/analysis/excluded_tasks.csv` atau `results/pilot_analysis/excluded_tasks.csv`. Tidak ada task yang boleh dikeluarkan untuk menyesuaikan hasil dengan File A.

## 9. Deviasi dari File B

1. File A memuat angka ilustratif bertanda dummy; angka tersebut tidak dipakai sebagai ekspektasi.
2. File B merevisi RQ3 menjadi cost-effectiveness; revisi ini mengalahkan rancangan korelasi karakteristik kode pada File A.
3. Pipeline lokal menjalankan CodaMOSA dan CoverUp; Pynguin dan MuTAP ditangani kolaborator.
4. Populasi CM hanya memiliki delapan stratum tertil non-kosong; penggunaan delapan stratum telah disetujui.
5. B2 sementara menggunakan `auto` yang sama sebagai B1 karena endpoint konfigurasi B2 hanya mengekspos `auto`. Deviasi ini dicatat di docs/protocol.md dan tidak memenuhi dua rambang File B.

## 10. Keterbatasan

- Identitas GPT-5.6 Sol pada endpoint `auto` adalah klaim penjual dan tidak diverifikasi.
- Perbandingan lintas dataset tidak berpasangan karena unit MT dan CM berbeda.
- Hasil final tidak boleh disimpulkan sebelum probe B2, pilot, sanity checks, dan run utama selesai.
- Efek kontaminasi data tetap mungkin karena benchmark bersifat publik.

## 11. Artefak Reproduksi

- `config/experiment.toml`: protokol mesin-baca (tanpa secrets)
- `experiment_control.ipynb`: kontrol dan observasi proses
- `scripts/setup.sh`: setup tunggal
- `scripts/run_experiment.py`: entry point tunggal dan resumable
- `results/module_characteristics.csv`, `results/cm_sample.csv`, `results/*matrix.csv`
- `results/raw/<run_id>/`: request/response teredaksi
- `results/generated/<run_id>/`: generated tests dan logs
- `results/analysis/`: tabel statistik, failure rate, task exclusion, dan deviations


## 3. Endpoint/tool probes — required before pilot

This validates B1/B2 `model_returned`, CoverUp function calling, and the per-call rate gate. The probe sends `tool_choice: required` (string form) because this endpoint rejects the object form. Both backbones currently route to the same `entah/auto` endpoint (documented File B deviation).`

In [14]:
probe_run = run([sys.executable, "scripts/probe_endpoints.py"], timeout=300, check=False)
probe = ROOT / "results" / "endpoint_probe.json"
probe_results = __import__("json").loads(probe.read_text(encoding="utf-8"))
for item in probe_results:
    print(item)
failed = [item for item in probe_results if not item.get("passed")]
PROBES_PASSED = not failed
if failed:
    print("\nBLOCKED: a backbone probe did not pass. Fix the failing backbone, rerun this cell, then continue to Section 4.")
    print("Probe report saved to results/endpoint_probe.json")
else:
    print("Both endpoint function-calling probes passed. Continue to pilot smoke test.")
PROBES_PASSED

$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/probe_endpoints.py
{
  "backbone": "B1",
  "passed": true,
  "model_requested": "entah/auto",
  "model_returned": "auto",
  "function_calling": true,
  "http_status": 200,
  "tokens_in": 507,
  "tokens_out": 11,
  "finish_reason": "tool_calls"
}
{
  "backbone": "B2",
  "passed": true,
  "model_requested": "entah/auto",
  "model_returned": "auto",
  "function_calling": true,
  "http_status": 200,
  "tokens_in": 507,
  "tokens_out": 11,
  "finish_reason": "tool_calls"
}
Both endpoint function-calling probes passed; wrote /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/results/endpoint_probe.json

exit=0 elapsed=0.8s
{'backbone': 'B1', 'passed': True, 'model_requested': 'entah/auto', 'model_returned': 'auto', 'function_calling': True, 'http_status': 200, 'tokens_in': 507, 'tokens_out': 11, 'finish_reason': 'tool_calls'}
{'backbone': 'B2', 'passed': True, 'model_requested': 'entah/auto', 'model_retur

True

## 4. Pilot — do not run until Section 3 passes

Runs 64 isolated jobs with checkpoints, 600 s budgets, 5 workers, raw logs, and the USD 900/14-day gates. This is a paid operation.

A smoke run (--max-runs 1) executes first to validate the full tool path before launching the remaining pilot jobs.

**Pause / resume:** The runner checkpoints every completed run to `results/pilot_results.csv`. To pause: `touch results/pause.requested` (or run `python scripts/run_experiment.py --pause`). The current in-flight run will finish, then no new runs launch. To resume: `python scripts/run_experiment.py --resume --retry-failed --pilot`. Re-running without `--retry-failed` skips already-completed runs automatically.`

In [15]:
PROBES_PASSED = globals().get("PROBES_PASSED", False)
CONFIRM_PAID_PILOT = True

if not PROBES_PASSED:
    print("SKIPPED: Section 3 probe gate has not passed. No pilot task was started.")
elif not CONFIRM_PAID_PILOT:
    print("SKIPPED: set CONFIRM_PAID_PILOT=True after reviewing probe results.")
else:
    # First run one task to observe the full CodaMOSA/CoverUp path before 64 pilot runs.
    try:
        run([sys.executable, "scripts/run_experiment.py", "--pilot", "--max-runs", "1", "--workers", "1"], timeout=2400)
        print_status()
    except RuntimeError as exc:
        print(f"\nSmoke run failed: {exc}")
        print("Aborting remaining pilot runs. Inspect results/generated/ and results/raw/ for the failure.")
        raise
    # If the smoke run is OK, resume all remaining pilot jobs.
    run([sys.executable, "scripts/run_experiment.py", "--pilot"], timeout=172800)

$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/run_experiment.py --pilot --max-runs 1 --workers 1
{
  "matrix": "/home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/results/pilot_matrix.csv",
  "results": "/home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/results/pilot_results.csv",
  "total": 64,
  "completed_ok": 0,
  "recorded": 64,
  "pending": 0,
  "cost_usd": 3.370008,
  "blocked_all": true
}

exit=0 elapsed=0.1s
{
  "credentials": {
    "LLM_API_KEY": true,
    "LLM_BASE_URL": true,
    "LLM_API_KEY_B2": true,
    "LLM_BASE_URL_B2": true
  },
  "files": {
    "pilot_matrix.csv": {
      "rows": 64,
      "ok": 0,
      "failed": 0,
      "pending": 64,
      "cost_usd": 0.0
    },
    "pilot_results.csv": {
      "rows": 64,
      "ok": 0,
      "failed": 64,
      "pending": 0,
      "cost_usd": 3.370008
    },
    "run_matrix.csv": {
      "rows": 8120,
      "ok": 0,
      "failed": 0,
      "pending": 8120,
      "cost_usd": 0.0


## 5. Pilot monitoring and analysis

Rerun while the pilot is active or after interruption. The runner is resumable — every completed run is checkpointed, so re-running skips what is already done.

**Resume commands:**
- `python scripts/run_experiment.py --resume --pilot` (skips completed runs)
- `python scripts/run_experiment.py --resume --retry-failed --pilot` (also re-runs failed runs)
- `python scripts/run_experiment.py --pause` (pause after current run finishes)`

In [16]:
print_status()
path = ROOT / "results" / "pilot_results.csv"
if path.exists():
    run([sys.executable, "scripts/analyze.py", str(path), "--output", "results/pilot_analysis"])
else:
    print("SKIPPED: no pilot_results.csv yet.")

{
  "credentials": {
    "LLM_API_KEY": true,
    "LLM_BASE_URL": true,
    "LLM_API_KEY_B2": true,
    "LLM_BASE_URL_B2": true
  },
  "files": {
    "pilot_matrix.csv": {
      "rows": 64,
      "ok": 0,
      "failed": 0,
      "pending": 64,
      "cost_usd": 0.0
    },
    "pilot_results.csv": {
      "rows": 64,
      "ok": 0,
      "failed": 64,
      "pending": 0,
      "cost_usd": 3.370008
    },
    "run_matrix.csv": {
      "rows": 8120,
      "ok": 0,
      "failed": 0,
      "pending": 8120,
      "cost_usd": 0.0
    }
  }
}
$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/analyze.py /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/results/pilot_results.csv --output results/pilot_analysis
Analysis written to results/pilot_analysis
                         n_total  n_failed error_category  count  fail_rate  over_10pct
suite tool     backbone                                                                
CM    CodaMOSA B1             

## 6. Full run gate

File B requires pilot timing/cost extrapolation and explicit approval before protocol freeze/full R=10. Approval has been given; this cell will launch the full R=10 run (8,120 jobs) after Section 5 completes.`

In [17]:
FULL_RUN_APPROVED = True
if not FULL_RUN_APPROVED:
    print("SKIPPED: full R=10 run needs completed pilot analysis and explicit approval.")
    print("Set FULL_RUN_APPROVED = True and rerun this cell when ready.")
else:
    run([sys.executable, "scripts/build_matrix.py"])
    run([sys.executable, "scripts/run_experiment.py"], timeout=1209600)

$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/build_matrix.py
Wrote 8120 runs to results/run_matrix.csv

exit=0 elapsed=0.1s
$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak-main/.venv/bin/python scripts/run_experiment.py


KeyboardInterrupt: 

## 7. Final analysis and figures

Generates Tables IV–IX, paired tests, cost-effectiveness, failure rate, exclusion audit, and Figures 3–5 from real `results.csv`. Runs after the full run completes.`

In [ ]:
results = ROOT / "results" / "results.csv"
if not results.exists():
    print("SKIPPED: no final results.csv yet.")
else:
    run([sys.executable, "scripts/analyze.py", str(results)])
    run([sys.executable, "scripts/figures.py", str(results)])
run([sys.executable, "scripts/report.py"])
print((ROOT / "experiment_report.md").read_text(encoding="utf-8"))